<a href="https://colab.research.google.com/github/yadavrishikesh/Masterclass-V2-2026/blob/main/module2/code/Lab_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lab: Image Classification with Neural Networks and CNNs
**Author:** Dr. Rishikesh Yadav / Vedant Vibhor

### Objectives
1. Understand images as numerical tensors (Refer Lecture: *Grayscale vs RGB*).
2. Build an MLP on flattened images and observe the **Parameter Explosion** (Refer Lecture: *Why not just flatten?*).
3. Manually calculate CNN output shapes using the lecture formula (Refer Lecture: *Key Hyperparameters in Convolution*).
4. Build a CNN and compare parameter efficiency and accuracy.
5. Perform Error Analysis (Confusion Matrix, Failure Analysis).
6. Implement advanced training strategies: **Batch Normalization**, **Dropout**, and **Early Stopping** (Refer Lecture: *Useful Strategies for Training*).

**Dataset:** [Fashion-MNIST](https://www.tensorflow.org/tutorials/keras/classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, BatchNormalization, Dropout, Activation

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

---
## Part 1: Images as Tensors
Computers "read" images as arrays of numbers. A grayscale image is a 2D matrix $I \in \mathbb{R}^{H \times W}$. RGB images are 3D tensors $X \in \mathbb{R}^{H \times W \times 3}$.

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

print("X_train shape:", X_train.shape)

### Exercise 1: Tensor Anatomy
1. Looking at the shape `(60000, 28, 28)`, what does each number represent?
2. Is this grayscale or RGB?
3. What is the maximum possible value of a pixel in this raw dataset (before normalization)?

In [ ]:
plt.figure(figsize=(8,8))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize pixel values to [0, 1] (Crucial for Neural Networks)
X_train = X_train / 255.0
X_test = X_test / 255.0

print("Min pixel value:", X_train.min())
print("Max pixel value:", X_train.max())

---
## Part 2: The MLP Approach & Parameter Explosion
From the lecture: *"Flattening a $64 \times 64 \times 3$ image to $d = 12,288$. A Dense layer with 1,024 units has $\approx 12.6$ million parameters. No weight sharing, no explicit locality."*

In [ ]:
mlp = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

mlp.summary()

### Exercise 2: Manual Parameter Calculation
Using the lecture formula:
1. Calculate the parameters for the first Dense layer manually. ($d = 28 \times 28$, $m = 128$)
2. Does your calculation match the `Param #` in the summary?

In [ ]:
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_mlp = mlp.fit(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)

---
## Part 3: The CNN Approach & Shape Tracking
Before running the code below, use the **Output Size Formula** from the lecture:
$$H' = \lfloor\frac{H + 2P - k}{S}\rfloor + 1$$

1. **Input:** (28, 28, 1)
2. **Conv2D(32, (3,3), padding='valid'):** $H' = \lfloor\frac{28 + 0 - 3}{1}\rfloor + 1 = ?$
3. **MaxPooling2D(2,2):** $H' = ?$
4. **Conv2D(64, (3,3), padding='valid'):** $H' = ?$
5. **MaxPooling2D(2,2):** $H' = ?$ (Note: 11/2 rounds down to 5)

In [ ]:
X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

cnn = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1), padding='valid'),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu', padding='valid'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(16, activation='relu'),
    Dense(10, activation='softmax')
])

cnn.summary()

# from tensorflow.keras.layers import GlobalAveragePooling2D

# cnn = Sequential([
#     Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1), padding='valid'),
#     MaxPooling2D((2,2)),
#     Conv2D(64, (3,3), activation='relu', padding='valid'),
#     MaxPooling2D((2,2)),

#     GlobalAveragePooling2D(),

#     Dense(64, activation='relu'),
#     Dense(10, activation='softmax')
# ])

# cnn.summary()

### Exercise 3: Architecture Analysis
1. Check the `Output Shape` column. Did it match your manual calculations?
2. Look at the `Param #` for the first `conv2d` layer. Prove it using the formula: $K=32, k=3, C=1$.
3. **Discussion:** The lecture mentions `padding='same'` keeps the size identical ($H' = H$). If we changed the first layer to `padding='same'`, what would the output shape be?

In [ ]:
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_cnn = cnn.fit(X_train_cnn, y_train, epochs=10, validation_split=0.1, verbose=1)

---
## Part 4: Training Diagnostics & Error Analysis

In [ ]:
def plot_learning_curve(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0].set_title(f'{title} - Loss'); axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Validation Acc')
    axes[1].set_title(f'{title} - Accuracy'); axes[1].legend()
    plt.show()

plot_learning_curve(history_mlp, "MLP")
plot_learning_curve(history_cnn, "CNN")

### Exercise 4: Curve Interpretation
Look at the Validation Loss of the CNN. Does it start to increase while Training Loss decreases? This is the textbook definition of **overfitting**.

In [ ]:
# Baseline & Test Eval Comparison
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_flat, y_train)
baseline_acc = accuracy_score(y_test, dummy.predict(X_test_flat))

mlp_loss, mlp_acc = mlp.evaluate(X_test, y_test, verbose=0)
cnn_loss, cnn_acc = cnn.evaluate(X_test_cnn, y_test, verbose=0)

print(f"Baseline Accuracy: {baseline_acc:.4f}")
print(f"MLP Test Accuracy: {mlp_acc:.4f}")
print(f"CNN Test Accuracy: {cnn_acc:.4f}")

### Confusion Matrix Analysis

In [ ]:
cnn_pred = cnn.predict(X_test_cnn, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, cnn_pred)

plt.figure(figsize=(10,8))
# Added class names to the axes and annotations to see exact numbers
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Exercise: Which classes are most frequently confused?
While the heatmap is useful, a Classification Report gives us exact metrics (Precision/Recall) per class.

In [ ]:
# TODO: Print the classification report for the CNN predictions
# Hint: use classification_report(y_test, cnn_pred, target_names=class_names)



### Failure Analysis

In [ ]:
wrong = np.where(cnn_pred != y_test)[0]

plt.figure(figsize=(10,10))
for i, idx in enumerate(wrong[:9]):
    plt.subplot(3,3,i+1)
    plt.imshow(X_test[idx], cmap="gray")
    plt.title(f"True: {class_names[y_test[idx]]}\nPred: {class_names[cnn_pred[idx]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Reflection Questions
1. Are the mistakes the model makes reasonable?
2. Would a human confuse these classes (e.g., Shirt vs T-shirt/top, Pullover vs Coat)?

In [ ]:
# Write your reflections here:
# 1.
# 2.


---
## Part 5: Advanced CNN (Regularization Strategies)
The lecture introduced three key strategies to fix the overfitting we just observed:
1. **Batch Normalization:** Stabilizes activations. Applied *after* Conv, *before* Activation (`Conv -> BatchNorm -> ReLU`).
2. **Dropout:** Randomly deactivates neurons to prevent overfitting.
3. **Early Stopping:** Stops training when validation loss stops improving.

In [ ]:
# Define the Early Stopping callback
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,            # Wait 3 epochs without improvement before stopping
    restore_best_weights=True # Keep the best model weights, not the last ones
)

In [ ]:
# TODO: Build an Advanced CNN
# 1. Remove 'activation="relu"' from the Conv2D layers.
# 2. Add BatchNormalization() followed by Activation('relu') for both conv blocks.
# 3. Add Dropout(0.3) after the Flatten layer.

advanced_cnn = Sequential([
    # Layer 1
    Conv2D(32, (3,3), padding='same', input_shape=(28,28,1)),
    # TODO: Add BatchNormalization
    # TODO: Add Activation('relu')
    MaxPooling2D((2,2)),

    # Layer 2
    Conv2D(64, (3,3), padding='same'),
    # TODO: Add BatchNormalization
    # TODO: Add Activation('relu')
    MaxPooling2D((2,2)),

    Flatten(),
    # TODO: Add Dropout(0.3)
    Dense(16, activation='relu'),
    Dense(10, activation='softmax')
])

advanced_cnn.summary()

In [ ]:
# TODO: Compile the advanced_cnn (optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



In [ ]:
# TODO: Train the advanced_cnn for 50 epochs.
# Crucial Step: Pass `callbacks=[early_stop]` to the .fit() function!
# Because of Early Stopping, it won't actually run all 50 epochs if it stops improving.

history_adv = None


In [ ]:
# Plot the learning curve for the advanced CNN
plot_learning_curve(history_adv, "Advanced CNN (BN + Dropout)")

# Evaluate
adv_loss, adv_acc = advanced_cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f"Advanced CNN Test Accuracy: {adv_acc:.4f}")

### Final Model Comparison

In [ ]:
final_comparison = pd.DataFrame({
    "Model": ["Baseline (Guess Most Frequent)", "Standard MLP", "Standard CNN", "Advanced CNN (+Reg)"],
    "Test Accuracy": [baseline_acc, mlp_acc, cnn_acc, adv_acc],
    "Total Parameters": [0, mlp.count_params(), cnn.count_params(), advanced_cnn.count_params()]
})

# Sort by accuracy to clearly see the winner
final_comparison.sort_values(by="Test Accuracy", ascending=False)

---
## Bonus: Visualizing Learned Filters
The lecture states: *"Convolution layer: learn local feature detectors (edges, textures, parts)."*

In [ ]:
# Extract filters from the first Conv2D layer of the standard CNN
filters, biases = cnn.layers[0].get_weights()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.imshow(filters[:, :, 0, i], cmap='gray')
    ax.axis('off')
plt.suptitle("First Layer Learned Kernels (Edge Detectors)")
plt.show()

## Bonus: Feature Map Visualization
What does the network "see" when it looks at an image?

In [ ]:
# Create a model that outputs the feature maps of the first layer
inp = tf.keras.Input(shape=(28, 28, 1))
out = cnn.layers[0](inp)
feature_model = tf.keras.Model(inputs=inp, outputs=out)

feature_maps = feature_model.predict(X_test_cnn[2:3], verbose=0)

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0,:,:,i], cmap="gray")
    ax.axis("off")
plt.suptitle("Feature Maps Activated by a Test Image")
plt.show()

### Key Takeaways
- Images are $H \times W \times C$ tensors.
- MLPs destroy spatial structure and suffer from parameter explosion ($d \times m$).
- CNNs preserve spatial structure via local connectivity and weight sharing.
- You can mathematically predict the output shape of a CNN layer using the formula.
- **Batch Normalization**, **Dropout**, and **Early Stopping** are essential tools to combat overfitting and stabilize deep networks.